# This Notebook is used to Evaluate the Localization Performance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import PIL
import json
from PIL import Image
import os

In [ ]:
def read_localization_data(folder_path):
    keyframe_time=[]
    query_id = []
    map_id = []
    loc_success = []
    comp_time = []
    temporal_depth = []
    window_num_vertices = []
    inliers = []

    file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
    print(f"Number of files: {file_count}")
    for i in range(0,file_count,1):
        filename =folder_path+f"/loc_{i:05d}.txt"
        matrix = []
        
        with open(filename, 'r') as file:
            for line in file:
                # Skip empty lines
                if line.startswith('keyframe_time:'):
                    time =int(line.split(": ")[1].strip())
                    keyframe_time.append(time)
                elif line.startswith('query_id:'):
                    q_id =int(line.split(": ")[1].strip())
                    query_id.append(q_id)
                elif line.startswith("map_id:"):
                    m_id =int(line.split(": ")[1].strip())
                    map_id.append(m_id)
                elif line.startswith("loc_success:"):
                    loc_stat =int(line.split(": ")[1].strip())
                    loc_success.append(loc_stat)
                elif line.startswith("comp_time:"):
                    comp_t =int(line.split(": ")[1].strip())
                    comp_time.append(comp_t)
                elif line.startswith("temporal_depth:"):
                    temp_depth =int(line.split(": ")[1].strip())
                    temporal_depth.append(temp_depth)
                elif line.startswith("window_num_vertices:"):
                    num_vert =int(line.split(": ")[1].strip())
                    window_num_vertices.append(num_vert)
                elif line.startswith("inliers:"):
                    inlier =int(line.split(": ")[1].strip())
                    inliers.append(inlier)
                elif line.strip() and not line.startswith('u'):
                    row = [float(val) for val in line.strip().split()]
                    matrix.append(row)
    data = {
    'query_id': np.array(query_id),
    'map_id': np.array(map_id),
    'keyframe_time': np.array(keyframe_time),
    'loc_success': np.array(loc_success),
    'comp_time': np.array(comp_time),
    'temporal_depth': np.array(temporal_depth),
    'window_num_vertices': np.array(window_num_vertices),
    'inliers': np.array(inliers)
    }
    return data

def split_arrays(arrays_dict, split_indices):
    result = {}
    
    # Sort indices to ensure proper splitting
    split_indices = sorted(split_indices)
    
    # Create boundary points: [0, idx1, idx2, ..., None]
    boundaries = [0] + split_indices + [None]
    
    # Split each array
    for name, array in arrays_dict.items():
        for i in range(len(boundaries) - 1):
            start = boundaries[i]
            end = boundaries[i + 1]
            
            # Create the split array name
            split_name = f"{name}_{i + 1}"
            
            # Slice and copy the array
            if end is None:
                result[split_name] = array[start:].copy()
            else:
                result[split_name] = array[start:end].copy()
    
    return result

In [ ]:
folder_path = '/home/adam/Desktop/CurrentBranch/src/main/src/vtr_db_extractor/loc_status'
data = read_localization_data(folder_path)

repeat = split_arrays(data, [10000])
print(repeat['inliers_1'][0])

time_c = []
start_time_c =0
first_frame= True
for time in repeat['keyframe_time_1']:
    if first_frame:
        start_time = time
        first_frame = False
        time_c.append(0.0)

    else:
        time_c.append((time-start_time)/1000000000)
time_c = np.array(time_c)

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(repeat['inliers_1'].size)
x1 = np.linspace(1, repeat['inliers_1'].size, repeat['inliers_1'].size)
x2 = np.linspace(1, repeat['inliers_2'].size, repeat['inliers_2'].size)
plt.plot(time_c, repeat['inliers_1'], linewidth=0.5,  label = 'Repeat 1')
# plt.plot(x2, repeat['inliers_2'], linewidth=0.5, label='Repeat 2')

plt.xlabel('duration [s]')
plt.ylabel('Inliers')
# plt.legend()
plt.title('Inlier Count')
plt.show()

In [ ]:
np.savetxt('inliers_repeat_12_pm.txt', np.column_stack([time_c, repeat['inliers_1']]))

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(15, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(repeat['inliers_1'].size)

plt.scatter(repeat['map_id_1'], repeat['inliers_1'], s = 0.5,   label = 'Repeat 1')
plt.scatter(repeat['map_id_2'], repeat['inliers_2'], s = 0.5, label='Repeat 2')

plt.xlabel('Vertex')
plt.ylabel('Inliers')
plt.legend()
plt.title('Inlier Count')
plt.show()

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(repeat['temporal_depth_1'].size)
x1 = np.linspace(1, repeat['temporal_depth_1'].size, repeat['temporal_depth_1'].size)
x2 = np.linspace(1, repeat['temporal_depth_2'].size, repeat['temporal_depth_2'].size)
plt.scatter(x1, repeat['temporal_depth_1'], s=0.8,  label = 'Repeat 1')
plt.scatter(x2, repeat['temporal_depth_2'], s=0.8, label='Repeat 2')

plt.xlabel('Frame')
plt.ylabel('Temporal Depth')
plt.legend()
plt.title('Temporal Depth')
plt.show()

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(repeat['window_num_vertices_1'].size)
x1 = np.linspace(1, repeat['window_num_vertices_1'].size, repeat['window_num_vertices_1'].size)
x2 = np.linspace(1, repeat['window_num_vertices_2'].size, repeat['window_num_vertices_2'].size)
plt.scatter(x1, repeat['window_num_vertices_1'], s=0.8,  label = 'Repeat 1')
plt.scatter(x2, repeat['window_num_vertices_2'], s=0.8, label='Repeat 2')

plt.xlabel('Frame')
plt.ylabel('window num vertices')
plt.legend()
plt.title('window num vertices')
plt.show()

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(repeat['loc_success_1'].size)
x1 = np.linspace(1, repeat['loc_success_1'].size, repeat['loc_success_1'].size)
x2 = np.linspace(1, repeat['loc_success_2'].size, repeat['loc_success_2'].size)
plt.scatter(time_c, repeat['loc_success_1'], s=0.8,  label = 'Repeat 1')
# plt.scatter(x2, repeat['loc_success_2'], s=0.8, label='Repeat 2')

plt.xlabel('Frame')
plt.ylabel('Loc Success')
plt.legend()
plt.title('Localization Success')
plt.show()

In [ ]:
np.savetxt('loc_status_repeat_OG_2_pm.txt', np.column_stack([time_c, repeat['loc_success_1']]))